In [2]:
import pandas as pd
import numpy as np

### Load Data + Cleaning

In [5]:
# Load the data
btc = pd.read_csv("BTC_full_data.csv")
eth = pd.read_csv("ETH_full_data.csv")

# Clean data (remove first row due to missing Log_Return value and convert cols to numeric)
def clean_data(data):
    df = data
    # remove first row
    df = df.iloc[1:].reset_index(drop=True)

    df["Date"] = pd.to_datetime(df["Date"])
    # convert numeric columns
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.set_index("Date")

    return df

btc = clean_data(btc)
eth = clean_data(eth)

print(btc.head())
print(eth.head())

                    Open          High           Low         Close  \
Date                                                                 
2018-01-02  13625.000000  15444.599609  13163.599609  14982.099609   
2018-01-03  14978.200195  15572.799805  14844.500000  15201.000000   
2018-01-04  15270.700195  15739.700195  14522.200195  15599.200195   
2018-01-05  15477.200195  17705.199219  15202.799805  17429.500000   
2018-01-06  17462.099609  17712.400391  16764.599609  17527.000000   

               Adj Close       Volume  Log_Return  
Date                                               
2018-01-02  14982.099609  16846600192    0.092589  
2018-01-03  15201.000000  16871900160    0.014505  
2018-01-04  15599.200195  21783199744    0.025858  
2018-01-05  17429.500000  23840899072    0.110945  
2018-01-06  17527.000000  18314600448    0.005578  
                  Open         High         Low        Close    Adj Close  \
Date                                                                

### Create breakout signals

In [6]:
# Create 20-day breakout signals
# Long when price > 20-day high
# Short when price < 20-day low

def breakout_signal(df):

    data = df.copy()

    # 20 day breakout levels
    data['high20'] = data['Close'].shift(1).rolling(20).max()
    data['low20'] = data['Close'].shift(1).rolling(20).min()

    # signal
    data['signal'] = 0
    data.loc[data['Close'] > data['high20'], 'signal'] = 1
    data.loc[data['Close'] < data['low20'], 'signal'] = -1

    # hold previous position
    data['position'] = data['signal'].replace(0, np.nan).ffill().fillna(0)

    return data

In [ ]:
btc = breakout_signal(btc)
eth = breakout_signal(eth)

#print(btc)

                    Open          High           Low         Close  \
Date                                                                 
2018-01-02  13625.000000  15444.599609  13163.599609  14982.099609   
2018-01-03  14978.200195  15572.799805  14844.500000  15201.000000   
2018-01-04  15270.700195  15739.700195  14522.200195  15599.200195   
2018-01-05  15477.200195  17705.199219  15202.799805  17429.500000   
2018-01-06  17462.099609  17712.400391  16764.599609  17527.000000   
...                  ...           ...           ...           ...   
2025-12-26  87235.507812  89459.429688  86628.140625  87301.429688   
2025-12-27  87301.429688  87874.781250  87182.976562  87802.156250   
2025-12-28  87799.343750  87986.890625  87394.953125  87835.835938   
2025-12-29  87835.789062  90299.156250  86717.914062  87138.140625   
2025-12-30  87134.351562  89297.937500  86735.546875  88430.132812   

               Adj Close       Volume  Log_Return        high20         low20  \
Date    

In [9]:
# Find Strategy Returns
btc['strategy_return'] = btc['position'].shift(1) * btc['Log_Return']
eth['strategy_return'] = eth['position'].shift(1) * eth['Log_Return']

### Evaluation
We will evaluate the strategy using 6 metrics:
1. Cumulative PnL
2. Average Daily PnL
3. Volatility
4. Annualised Return
5. Sharpe Ratio
6. Max Drawdown

In [10]:
def evaluate_strategy(strategy_returns):
    r = strategy_returns.dropna()

    # cumulative pnl
    cumulative_pnl = np.exp(r.cumsum()).iloc[-1] - 1
    # average daily pnl
    avg_daily_pnl = r.mean()
    # volatility
    volatility = r.std()
    # annualised return
    annual_return = r.mean() * 365
    # annualised volatility
    annual_vol = volatility * np.sqrt(365)
    # sharpe ratio
    rf = 0.03
    sharpe = (annual_return - rf) / annual_vol
    # max drawdown
    cumulative = np.exp(r.cumsum())
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()

    return {
        "Cumulative PnL": cumulative_pnl,
        "Average Daily PnL": avg_daily_pnl,
        "Volatility": volatility,
        "Annualised Return": annual_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }


In [11]:
# Run evaluation
btc_results = evaluate_strategy(btc['strategy_return'])
eth_results = evaluate_strategy(eth['strategy_return'])

print("BTC Results")
print(btc_results)

print("\nETH Results")
print(eth_results)

BTC Results
{'Cumulative PnL': np.float64(45.95817178190562), 'Average Daily PnL': np.float64(0.0013186903882739158), 'Volatility': np.float64(0.03367532591850597), 'Annualised Return': np.float64(0.48132199171997925), 'Sharpe Ratio': np.float64(0.7015009381913062), 'Max Drawdown': np.float64(-0.5104578324177506)}

ETH Results
{'Cumulative PnL': np.float64(118.41971491748393), 'Average Daily PnL': np.float64(0.0016384529989346046), 'Volatility': np.float64(0.04417113015424797), 'Annualised Return': np.float64(0.5980353446111307), 'Sharpe Ratio': np.float64(0.6731167783300379), 'Max Drawdown': np.float64(-0.8547008874835758)}


### Results Interpretation
BTC:
1. Cumulative PnL: 45.96 -> The portfolio grew 45.96 times its initial value
2. Average Daily PnL: 0.132% avarage return per day
3. Volatility: 3.37%
4. Annualised Return: 0.48 -> 48% per year
5. Sharpe Ratio: 0,70
6. Max Drawdown: -0.51

ETH:
1. Cumulative PnL: 118.42 -> The portfolio grew 118.42 times its initial value
2. Average Daily PnL: 0.164% avarage return per day
3. Volatility: 4.42%
4. Annualised Return: 0.60 -> 60% per year
5. Sharpe Ratio: 0,67
6. Max Drawdown: -0.85

